# Evaluation & Export

In [1]:
# ---------------------------------------------------------
# Section 1: Setup & Imports
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import os
import pathlib
import joblib
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn Evaluation Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    recall_score, f1_score, roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

# TensorFlow / Keras (for loading the LSTM)
import tensorflow as tf

# MLflow
import mlflow

# --- MLflow Setup ---
# Point exactly to the sandboxed tracking database we created in Notebook 2
mlflow.set_tracking_uri("sqlite:///../mlflow/mlruns.db")
experiment_name = "bluecradle_training"
mlflow.set_experiment(experiment_name)

print("Libraries imported.")
print(f"Connected to MLflow Tracking URI: {mlflow.get_tracking_uri()}")

# --- Load Test Data ---
data_dir = "../data/prepared_data" # Adjusting relative path just in case

# 1. DHS Test Data (for RF and XGBoost)
X_dhs_test = pd.read_csv(f"{data_dir}/X_dhs_test.csv")
y_dhs_test = pd.read_csv(f"{data_dir}/y_dhs_test.csv")['label']

# 2. MAL-ED Test Tensors (for LSTM)
maled_data = np.load(f"{data_dir}/maled_tensors.npz", allow_pickle=True)
X_test_seq = maled_data['X_test_seq']
X_test_static = maled_data['X_test_static']
y_test_seq = maled_data['y_test_seq']

print("\n--- Test Data Loaded ---")
print(f"DHS Test Features: {X_dhs_test.shape} | Targets: {y_dhs_test.shape}")
print(f"MAL-ED Test Sequences: {X_test_seq.shape} | Static: {X_test_static.shape} | Targets: {y_test_seq.shape}")

Libraries imported.
Connected to MLflow Tracking URI: sqlite:///../mlflow/mlruns.db

--- Test Data Loaded ---
DHS Test Features: (999, 18) | Targets: (999,)
MAL-ED Test Sequences: (140, 6, 8) | Static: (140, 3) | Targets: (140, 6)


## Load All Runs from MLflow

In [2]:
# ---------------------------------------------------------
# Section 2: Load All Runs from MLflow
# ---------------------------------------------------------
from IPython.display import display

print("Fetching runs from MLflow...")

# 1. Get the current active experiment
experiment = mlflow.get_experiment_by_name(experiment_name)

# 2. Pull all runs into a Pandas DataFrame
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# 3. Filter down to the exact columns we need for evaluation
cols_to_keep = [
    'run_id',
    'tags.mlflow.runName', 
    'metrics.sam_recall', 
    'metrics.mam_recall', 
    'metrics.f1_weighted',
    'metrics.accuracy'
]

# Ensure we only try to pull columns that exist
existing_cols = [col for col in cols_to_keep if col in runs_df.columns]
summary_df = runs_df[existing_cols].copy()

# 4. Clean up column names for readability
summary_df.rename(columns={
    'run_id': 'Run ID',
    'tags.mlflow.runName': 'Model Architecture',
    'metrics.sam_recall': 'SAM Recall',
    'metrics.mam_recall': 'MAM Recall',
    'metrics.f1_weighted': 'Weighted F1',
    'metrics.accuracy': 'Overall Accuracy'
}, inplace=True)

# 5. Sort by our primary clinical metric: SAM Recall (descending)
summary_df = summary_df.sort_values(by='SAM Recall', ascending=False).reset_index(drop=True)

# 6. Display the final leaderboard
print("\n--- BlueCradle Model Leaderboard ---")
display(summary_df)

# Store the run IDs in a dictionary for easy access in Section 3
run_ids = dict(zip(summary_df['Model Architecture'], summary_df['Run ID']))

Fetching runs from MLflow...

--- BlueCradle Model Leaderboard ---


,Run ID,Model Architecture,SAM Recall,MAM Recall,Weighted F1,Overall Accuracy
0,4d93b74d05f344c88cc019160a153d66,xgboost,1.000000,1.000000,1.000000,1.000000
1,772b43547d374f18ad3df2d60a7b2ee6,xgboost,1.000000,1.000000,1.000000,1.000000
2,ad6dcca031304718891a021499acab07,random_forest_baseline,0.992958,1.000000,0.993094,0.992993
3,70b9d023d3c74da8ba939e878c06ddc2,random_forest_baseline,0.992958,1.000000,0.993094,0.992993
4,0ac33f699a504c31b2982138273a07e5,lstm_feedforward_hybrid,0.824859,0.957627,0.900698,0.893819
5,0c9127fffc8c4b51b765b4ab786f07fa,lstm_feedforward_hybrid,0.745763,0.966102,0.879043,0.870048


## The Evaluation Plotting Engine

In [3]:
# ---------------------------------------------------------
# Section 3.1: Deep Evaluation Plotting Engine
# ---------------------------------------------------------
def evaluate_and_log_model(y_true, y_probs, run_id, model_name, class_names=['Normal', 'MAM', 'SAM']):
    """Generates and logs deep evaluation plots to an existing MLflow run."""
    
    y_pred = np.argmax(y_probs, axis=1)
    
    # Wrap in np.asarray to guarantee a dense array and fix the "spmatrix" warning
    y_true_bin = np.asarray(label_binarize(y_true, classes=[0, 1, 2]))
    n_classes = y_true_bin.shape[1]

    # Re-open the specific MLflow run to attach these new artifacts
    with mlflow.start_run(run_id=run_id):
        print(f"Generating deep evaluation artifacts for: {model_name}...")
        
        # 1. Normalized Confusion Matrix
        fig_cm, ax_cm = plt.subplots(figsize=(7, 5))
        cm = confusion_matrix(y_true, y_pred, normalize='true')
        sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax_cm)
        ax_cm.set_title(f"{model_name} - Normalized Confusion Matrix")
        ax_cm.set_xlabel("Predicted")
        ax_cm.set_ylabel("True")
        cm_path = f"{model_name}_norm_cm.png"
        fig_cm.savefig(cm_path, bbox_inches='tight')
        mlflow.log_artifact(cm_path)
        plt.close(fig_cm)

        # 2. Precision, Recall, F1 Bar Chart per Class
        report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
        
        # Use .get() to safely parse the dictionary and satisfy strict type checking
        metrics_df = pd.DataFrame({
            'Class': class_names,
            'Precision': [float(report.get(c, {}).get('precision', 0.0)) for c in class_names],
            'Recall': [float(report.get(c, {}).get('recall', 0.0)) for c in class_names],
            'F1-Score': [float(report.get(c, {}).get('f1-score', 0.0)) for c in class_names]
        }).set_index('Class')

        fig_bar, ax_bar = plt.subplots(figsize=(8, 5))
        metrics_df.plot(kind='bar', ax=ax_bar, colormap='viridis', edgecolor='black')
        ax_bar.set_title(f"{model_name} - Class-wise Metrics")
        ax_bar.set_ylim((0.0, 1.1)) # Changed to tuple
        plt.xticks(rotation=0)
        ax_bar.legend(loc='lower right')
        bar_path = f"{model_name}_class_metrics.png"
        fig_bar.savefig(bar_path, bbox_inches='tight')
        mlflow.log_artifact(bar_path)
        plt.close(fig_bar)

        # 3. ROC Curves (One-vs-Rest)
        fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
            roc_auc = auc(fpr, tpr)
            ax_roc.plot(fpr, tpr, lw=2, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
        
        ax_roc.plot([0, 1], [0, 1], 'k--', lw=2)
        ax_roc.set_xlim((0.0, 1.0)) # Changed to tuple
        ax_roc.set_ylim((0.0, 1.05)) # Changed to tuple
        ax_roc.set_xlabel('False Positive Rate')
        ax_roc.set_ylabel('True Positive Rate')
        ax_roc.set_title(f"{model_name} - Multi-class ROC Curve")
        ax_roc.legend(loc="lower right")
        roc_path = f"{model_name}_roc_curve.png"
        fig_roc.savefig(roc_path, bbox_inches='tight')
        mlflow.log_artifact(roc_path)
        plt.close(fig_roc)

        # 4. Precision-Recall Curves (Crucial for imbalanced SAM class)
        fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
        for i in range(n_classes):
            precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_probs[:, i])
            avg_pr = average_precision_score(y_true_bin[:, i], y_probs[:, i])
            ax_pr.plot(recall, precision, lw=2, label=f'{class_names[i]} (AP = {avg_pr:.2f})')
            
        ax_pr.set_xlim((0.0, 1.0)) # Changed to tuple
        ax_pr.set_ylim((0.0, 1.05)) # Changed to tuple
        ax_pr.set_xlabel('Recall')
        ax_pr.set_ylabel('Precision')
        ax_pr.set_title(f"{model_name} - Precision-Recall Curve")
        ax_pr.legend(loc="lower left")
        pr_path = f"{model_name}_pr_curve.png"
        fig_pr.savefig(pr_path, bbox_inches='tight')
        mlflow.log_artifact(pr_path)
        plt.close(fig_pr)
        
        # Cleanup local images
        for f in [cm_path, bar_path, roc_path, pr_path]:
            if os.path.exists(f): os.remove(f)
            
print("Plotting engine initialized.")

Plotting engine initialized.


## Model Execution

In [4]:
# ---------------------------------------------------------
# Section 3.2: Execute Deep Evaluation per Model
# ---------------------------------------------------------

# --- 1. Random Forest Evaluation ---
rf_id = run_ids.get('random_forest_baseline')
if rf_id:
    # Load model from MLflow artifact store (ignoring strict dynamic typing)
    rf_model = mlflow.sklearn.load_model(f"runs:/{rf_id}/random_forest_model") # type: ignore
    
    # Get probabilities for ROC/PR curves
    rf_probs = rf_model.predict_proba(X_dhs_test) # type: ignore
    evaluate_and_log_model(y_dhs_test, rf_probs, rf_id, "Random_Forest")

# --- 2. XGBoost Evaluation ---
xgb_id = run_ids.get('xgboost')
if xgb_id:
    xgb_model = mlflow.xgboost.load_model(f"runs:/{xgb_id}/xgboost_model") # type: ignore
    xgb_probs = xgb_model.predict_proba(X_dhs_test) # type: ignore
    evaluate_and_log_model(y_dhs_test, xgb_probs, xgb_id, "XGBoost")

# --- 3. LSTM Hybrid Evaluation ---
lstm_id = run_ids.get('lstm_feedforward_hybrid')
if lstm_id:
    # Keras models can be heavy, suppress TF warnings during load
    tf.get_logger().setLevel('ERROR')
    lstm_model = mlflow.keras.load_model(f"runs:/{lstm_id}/lstm_hybrid_model") # type: ignore
    
    # Predict returns a 3D probability array for sequences
    lstm_probs_3d = lstm_model.predict([X_test_seq, X_test_static], verbose=0) # type: ignore
    
    # We must rigorously mask out the -1 padding to avoid skewing the evaluation
    valid_mask = y_test_seq != -1
    lstm_y_true_flat = y_test_seq[valid_mask]
    lstm_probs_flat = lstm_probs_3d[valid_mask]
    
    evaluate_and_log_model(lstm_y_true_flat, lstm_probs_flat, lstm_id, "LSTM_Hybrid")

print("\n✅ Deep Evaluation complete! All curves logged to MLflow artifacts.")

Generating deep evaluation artifacts for: Random_Forest...
Generating deep evaluation artifacts for: XGBoost...
Generating deep evaluation artifacts for: LSTM_Hybrid...

✅ Deep Evaluation complete! All curves logged to MLflow artifacts.


## Final Model Selection & Export

In [5]:
# ---------------------------------------------------------
# Section 4 & 5: Final Model Selection & Export
# ---------------------------------------------------------
import shutil
import joblib

print("🏆 --- Final Model Selection --- 🏆")

# 1. Human-in-the-Loop Selection Override
# While XGBoost/RF achieved near 1.0 metrics, this indicates target leakage due to 
# the cross-sectional nature of DHS data (diagnostic features like WHZ/MUAC). 
# For BlueCradle's preventative early-warning use case, the LSTM is the scientifically 
# valid choice as it models true longitudinal forecasting.

winning_model = "lstm_feedforward_hybrid"

# Extract the LSTM's specific row and Run ID
lstm_row = summary_df[summary_df['Model Architecture'] == winning_model].iloc[0]
winning_run_id = lstm_row['Run ID']

print(f"Winner deliberately selected: {winning_model}")
print(f"Rationale: Longitudinal Forecaster (Avoids DHS Target Leakage)")
print(f"SAM Recall: {lstm_row['SAM Recall']:.4f}")
print(f"MAM Recall: {lstm_row['MAM Recall']:.4f}")
print(f"Weighted F1: {lstm_row['Weighted F1']:.4f}\n")

# 2. Setup Export Directory
export_dir = pathlib.Path("../models").resolve()
export_dir.mkdir(exist_ok=True)
print(f"Export directory ready at: {export_dir}")

# 3. Export the Winning LSTM Model for Django
print("Exporting model artifacts...")

tf.get_logger().setLevel('ERROR')
model = mlflow.keras.load_model(f"runs:/{winning_run_id}/lstm_hybrid_model") # type: ignore
export_path = export_dir / "bluecradle_lstm_v1.keras"
model.save(export_path)

print(f"✅ Saved Deep Learning model to: {export_path.name}")
print("\nModel export complete! Ready for backend integration.")

🏆 --- Final Model Selection --- 🏆
Winner deliberately selected: lstm_feedforward_hybrid
Rationale: Longitudinal Forecaster (Avoids DHS Target Leakage)
SAM Recall: 0.8249
MAM Recall: 0.9576
Weighted F1: 0.9007

Export directory ready at: D:\final-year-project-team_bluecradle-ml-system\models
Exporting model artifacts...
✅ Saved Deep Learning model to: bluecradle_lstm_v1.keras

Model export complete! Ready for backend integration.


# Section 6: Django Integration Specification

To deploy this model into the BlueCradle backend API, the Django endpoint must strictly adhere to the following data pipeline schema:

### 1. Expected Input Payload (JSON)
The endpoint expects a flattened JSON object representing either a single clinical visit (for RF/XGBoost) or a time-series payload of 6 visits (for the LSTM). All categorical variables (e.g., `sex`, `breastfeeding_status`) must be numerically encoded prior to inference.

### 2. Required Preprocessing Steps
1.  **Imputation:** Missing numerical values (like `birth_weight`) must be filled using the **MICE Imputer** (`imputer.pkl`) fitted on the training data.
2.  **Scaling:** Continuous features must be standardized using the **StandardScaler** (`scaler.pkl`) to match the exact distribution the model was trained on. 

### 3. Output Mapping
The model returns a probability array (e.g., `[0.10, 0.25, 0.65]`). The Django view must extract `np.argmax(predictions)` and map it to the clinical definitions:
* `0` -> **Normal** (Standard regional monitoring)
* `1` -> **MAM** (Moderate Acute Malnutrition - Alert PHM)
* `2` -> **SAM** (Severe Acute Malnutrition - Immediate clinical intervention)

---

# Section 7: Final Pipeline Summary & Thesis Conclusion

The BlueCradle machine learning pipeline successfully progressed from raw demographic/clinical data to a deployment-ready predictive model. 

* **Architecture Choice:** We evaluated classical diagnostic trees (Random Forest, XGBoost) against a longitudinal forecasting deep learning model (LSTM + Feedforward Hybrid).
* **Evaluation Metric:** Priority was given strictly to **SAM Recall** to minimize false negatives for severe cases, followed by MAM Recall.
* **Limitations & Future Work:** The current models exhibit varying degrees of data leakage due to the inherent cross-sectional nature of DHS target definitions (WHZ/MUAC). When deployed regionally, this model should be fine-tuned via continuous learning as true, localized Sri Lankan infant growth trajectories are accumulated by the system over time.